# Generate Cross-Experiment Correctness Summary

This notebook creates a single CSV comparing whether each experiment solved each test example.

Output columns:

- `task_id`
- `difficulty`
- `hint`
- `example_number`
- `0_baseline_correct`
- `1_program_only_correct`
- `2_hypo_program_correct`
- `3_hint_hypo_program_correct`
- `4_good_hypo_program_correct`

The notebook reads from `final_outputs/*/trace.json` and writes:

```text
final_outputs/experiment_correctness_summary.csv
```

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Make the notebook runnable from either the repo root or the notebooks directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "final_outputs").exists():
    REPO_ROOT = REPO_ROOT.parent

PART1_SCRIPT_DIR = REPO_ROOT / "scripts" / "part1_hypogeneration"
sys.path.insert(0, str(PART1_SCRIPT_DIR))

from trace_to_csv import (  # noqa: E402
    _load_json,
    _load_task_metadata,
    _load_test_examples,
    convert_trace,
)

FINAL_OUTPUTS_DIR = REPO_ROOT / "final_outputs"
METADATA_PATH = REPO_ROOT / "data" / "task_data" / "train_hints_with_difficulty.csv"
ARC_CSV_PATH = REPO_ROOT / "data" / "task_data" / "ARC_training_tasks.csv"
OUTPUT_CSV_PATH = FINAL_OUTPUTS_DIR / "experiment_correctness_summary.csv"

EXPERIMENTS = [
    ("0_output_grid", "0_baseline_correct"),
    ("1_program_only", "1_program_only_correct"),
    ("2_hypo_program", "2_hypo_program_correct"),
    ("3_hint_hypo_program", "3_hint_hypo_program_correct"),
    ("4_good_hypo_program", "4_good_hypo_program_correct"),
]

print(f"Repo root: {REPO_ROOT}")
print(f"Output CSV: {OUTPUT_CSV_PATH}")

Repo root: /Users/liubeisong/Desktop/2026_Spring/CCM/FT-Hypothesis-Generation
Output CSV: /Users/liubeisong/Desktop/2026_Spring/CCM/FT-Hypothesis-Generation/final_outputs/experiment_correctness_summary.csv


## Load And Normalize Experiment Results

`trace_to_csv.py` already knows how to normalize the different trace schemas. This notebook reuses that logic and keeps only the top-1 selected/test correctness for each experiment.

In [2]:
def normalize_bool(value):
    """Return True/False for correctness values.

    Missing correctness in these selected-test rows means the selected program did
    not produce a scored match, so we count it conservatively as incorrect.
    """
    if isinstance(value, bool):
        return value
    if value is None or value == "":
        return False
    text = str(value).strip().lower()
    if text == "true":
        return True
    if text == "false":
        return False
    return False


metadata = _load_task_metadata(METADATA_PATH)
test_examples = _load_test_examples(ARC_CSV_PATH)

base_rows = {}
correctness_by_experiment = {}

for experiment_dir, correctness_col in EXPERIMENTS:
    trace_path = FINAL_OUTPUTS_DIR / experiment_dir / "trace.json"
    data = _load_json(trace_path)
    rows, _, trace_type = convert_trace(
        data,
        view="test",
        metadata=metadata,
        test_examples=test_examples,
    )
    df = pd.DataFrame(rows)

    # Direct-output test view has one row per candidate. For this summary,
    # use the first generated candidate, matching the top-1 baseline metric.
    if trace_type == "direct_output":
        df = df[df["is_top1"].astype(str).str.lower() == "true"].copy()

    df["example_number"] = pd.to_numeric(df["example_number"], errors="coerce").astype("Int64")
    df[correctness_col] = df["correct"].map(normalize_bool)

    for row in df.itertuples(index=False):
        key = (row.task_id, int(row.example_number))
        if key not in base_rows:
            base_rows[key] = {
                "task_id": row.task_id,
                "difficulty": getattr(row, "difficulty", ""),
                "hint": getattr(row, "hint", ""),
                "example_number": int(row.example_number),
            }

    correctness_by_experiment[correctness_col] = df.set_index(["task_id", "example_number"])[correctness_col]
    print(f"{experiment_dir}: trace_type={trace_type}, rows={len(df)}")

0_output_grid: trace_type=direct_output, rows=105
1_program_only: trace_type=direct_program, rows=105
2_hypo_program: trace_type=hypothesis_program, rows=105
3_hint_hypo_program: trace_type=hypothesis_program, rows=105
4_good_hypo_program: trace_type=hypothesis_program, rows=105


## Build And Save The Summary CSV

In [3]:
summary_df = pd.DataFrame(base_rows.values())
summary_df = summary_df.sort_values(["task_id", "example_number"]).reset_index(drop=True)

for _, correctness_col in EXPERIMENTS:
    summary_df[correctness_col] = [
        correctness_by_experiment[correctness_col].get((row.task_id, row.example_number), pd.NA)
        for row in summary_df.itertuples(index=False)
    ]

summary_df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Wrote {len(summary_df)} rows to {OUTPUT_CSV_PATH}")
summary_df.head()

Wrote 105 rows to /Users/liubeisong/Desktop/2026_Spring/CCM/FT-Hypothesis-Generation/final_outputs/experiment_correctness_summary.csv


,task_id,difficulty,hint,example_number,0_baseline_correct,1_program_only_correct,2_hypo_program_correct,3_hint_hypo_program_correct,4_good_hypo_program_correct
0,025d127b,Easy,Pay attention to which blocks stay fixed and w...,1,True,False,False,False,False
1,0a938d79,Easy,"The blocks’ initial positions and spacing, as ...",1,False,False,False,False,True
2,0b148d64,Easy,"Overall, it is actually divided into several l...",1,False,True,False,False,True
3,0d3d703e,Easy,The colors have a corresponding pattern of cha...,1,False,True,True,False,True
4,0dfd9992,Medium,Fill in the missing spaces. The whole image is...,1,False,False,True,True,True


## Quick Sanity Check

## Experiment-Level Accuracy Summary

This table summarizes each experiment with two metrics:

- `test_examples_accuracy`: fraction of individual test examples predicted correctly.
- `task_accuracy`: fraction of tasks where all test examples for that task are correct.

In [4]:
experiment_names = {
    "0_baseline_correct": "0_baseline",
    "1_program_only_correct": "1_program_only",
    "2_hypo_program_correct": "2_hypo_program",
    "3_hint_hypo_program_correct": "3_hint_hypo_program",
    "4_good_hypo_program_correct": "4_good_hypo_program",
}

accuracy_rows = []
for _, correctness_col in EXPERIMENTS:
    per_example = summary_df[correctness_col].astype(bool)
    per_task = summary_df.assign(_correct=per_example).groupby("task_id")["_correct"].all()

    accuracy_rows.append(
        {
            "experiment": experiment_names[correctness_col],
            "test_examples_accuracy": per_example.mean(),
            "task_accuracy": per_task.mean(),
            "test_examples_correct": f"{int(per_example.sum())}/{len(per_example)}",
            "tasks_correct": f"{int(per_task.sum())}/{len(per_task)}",
        }
    )

accuracy_summary_df = pd.DataFrame(accuracy_rows)
accuracy_summary_df

,experiment,test_examples_accuracy,task_accuracy,test_examples_correct,tasks_correct
0,0_baseline,0.133333,0.11,14/105,11/100
1,1_program_only,0.323810,0.30,34/105,30/100
2,2_hypo_program,0.257143,0.25,27/105,25/100
3,3_hint_hypo_program,0.409524,0.38,43/105,38/100
4,4_good_hypo_program,0.828571,0.82,87/105,82/100


In [5]:
summary_df[[col for _, col in EXPERIMENTS]].apply(lambda col: col.value_counts(dropna=False))

,0_baseline_correct,1_program_only_correct,2_hypo_program_correct,3_hint_hypo_program_correct,4_good_hypo_program_correct
False,91,71,78,62,18
True,14,34,27,43,87
